# Notebook — Order Block Reaction Strategy (Crypto)
## استراتژی واکنش به Order Block — Crypto (M5)

---

### تفاوت با نسخه فارکس (طلا/نقره)

| ویژگی | فارکس (XAUUSD) | Crypto (BTCUSDT) |
|-------|---------------|------------------|
| کمیسیون | $/lot ثابت یا spread pip | **درصد از notional value** |
| spread | pip-based | maker/taker order type |
| واحد پوزیشن | Lot (0.01, 0.1, 1) | Quantity (0.001 BTC) |
| محاسبه fee | ثابت/ترید | متغیر — بستگی به قیمت خروج |

### مدل کمیسیون
- **Maker** (Limit Order): `0.02%` از notional
- **Taker** (Market Order): `0.055%` از notional

### سناریوهای مقایسه
| سناریو | Entry | Exit | کل RT |
|--------|-------|------|-------|
| Maker-Maker | 0.02% | 0.02% | 0.04% |
| Maker-Taker | 0.02% | 0.055% | 0.075% |
| Taker-Taker | 0.055% | 0.055% | 0.11% |

---

| پارامتر | مقدار |
|---|---|
| نماد | BTCUSDT (قابل تغییر) |
| تایم‌فریم | M5 |
| Displacement Candles | ≥ 4 |
| Risk/Reward | 1:2 |

## Step 1 — Imports & Configuration

In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import List
from dataclasses import dataclass

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Symbol & Paths ───────────────────────────────────────────────────────────
SYMBOL        = 'BTCUSDT'   # change to ETHUSDT, SOLUSDT, etc.
DATA_DIR      = Path('./data')
RESULT_DIR    = Path('./result')
LOOKBACK_DAYS = 30

# ── Crypto Fee Structure ──────────────────────────────────────────────────────
MAKER_FEE = 0.0002    # 0.02%  — limit orders
TAKER_FEE = 0.00055   # 0.055% — market orders

# Three fee scenarios based on order type used
FEE_SCENARIOS = {
    'Maker-Maker': {'entry_pct': MAKER_FEE,  'exit_pct': MAKER_FEE},    # both limit
    'Maker-Taker': {'entry_pct': MAKER_FEE,  'exit_pct': TAKER_FEE},    # limit in, market out
    'Taker-Taker': {'entry_pct': TAKER_FEE,  'exit_pct': TAKER_FEE},    # both market
}

# ── Position Sizing ───────────────────────────────────────────────────────────
QUANTITY        = 0.001    # base quantity in coin (e.g. 0.001 BTC)
INITIAL_CAPITAL = 10_000   # USDT

# ── Order Block Detection ─────────────────────────────────────────────────────
DISPLACEMENT_MIN_CANDLES = 4
DISPLACEMENT_ATR_MULT    = 1.5
ATR_PERIOD               = 14
OB_EXPIRY_BARS           = 100

# ── Entry Conditions ──────────────────────────────────────────────────────────
REJECTION_WICK_RATIO = 0.3

# ── Risk ──────────────────────────────────────────────────────────────────────
RISK_REWARD    = 2.0
SL_BUFFER      = 0.0    # crypto prices are tight; set to price * 0.0001 if needed
MAX_TRADE_BARS = 288    # 24h in M5

print(f'Crypto Order Block Strategy — {SYMBOL}')
print(f'  Maker fee : {MAKER_FEE*100:.3f}%')
print(f'  Taker fee : {TAKER_FEE*100:.3f}%')
print(f'  Quantity  : {QUANTITY} coins')
print(f'  Capital   : ${INITIAL_CAPITAL:,} USDT')

Crypto Order Block Strategy — BTCUSDT
  Maker fee : 0.020%
  Taker fee : 0.055%
  Quantity  : 0.001 coins
  Capital   : $10,000 USDT


## Step 2 — Load Data

In [4]:
def load_ohlcv(symbol: str, tf: str, lookback_days: int = LOOKBACK_DAYS) -> pd.DataFrame:
    path = DATA_DIR / symbol / tf / 'ohlcv.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time'], utc=True)
    df = df.sort_values('time').reset_index(drop=True)
    keep = ['time', 'open', 'high', 'low', 'close', 'tick_volume', 'volume']
    df = df[[c for c in keep if c in df.columns]].copy()
    df.rename(columns={'tick_volume': 'volume'}, inplace=True)
    cutoff = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=lookback_days)
    return df[df['time'] >= cutoff].copy().reset_index(drop=True)


df_m5 = load_ohlcv(SYMBOL, 'M5')

print(f'M5 bars : {len(df_m5):,}  [{df_m5["time"].min()} → {df_m5["time"].max()}]')
print(f'Price range : ${df_m5["low"].min():,.2f} — ${df_m5["high"].max():,.2f}')
print(f'Last close  : ${df_m5["close"].iloc[-1]:,.2f}')
display(df_m5.tail(3))

FileNotFoundError: Missing: data\BTCUSDT\M5\ohlcv.csv

## Step 3 — ATR & Candle Analysis

In [ ]:
def add_candle_features(df: pd.DataFrame, atr_period: int = ATR_PERIOD) -> pd.DataFrame:
    df = df.copy()
    df['tr'] = np.maximum(
        df['high'] - df['low'],
        np.maximum(
            abs(df['high'] - df['close'].shift(1)),
            abs(df['low']  - df['close'].shift(1))
        )
    )
    df['atr']        = df['tr'].ewm(span=atr_period, adjust=False).mean()
    df['body']       = abs(df['close'] - df['open'])
    df['body_top']   = df[['open', 'close']].max(axis=1)
    df['body_bot']   = df[['open', 'close']].min(axis=1)
    df['upper_wick'] = df['high'] - df['body_top']
    df['lower_wick'] = df['body_bot'] - df['low']
    df['range']      = df['high'] - df['low']
    df['is_bull']    = df['close'] > df['open']
    df['is_bear']    = df['close'] < df['open']
    return df


df_m5 = add_candle_features(df_m5)
print(f'ATR (current) : {df_m5["atr"].iloc[-1]:,.2f}')
print(f'Avg Body      : {df_m5["body"].mean():,.2f}')
print(f'Avg Range     : {df_m5["range"].mean():,.2f}')

## Step 4 — Displacement Detection

In [ ]:
@dataclass
class Displacement:
    start_idx: int
    end_idx: int
    direction: str
    total_move: float
    candle_count: int
    start_price: float
    end_price: float
    start_time: object
    end_time: object


def detect_displacements(df, min_candles=DISPLACEMENT_MIN_CANDLES, atr_mult=DISPLACEMENT_ATR_MULT):
    displacements = []
    n = len(df)
    i = 0
    while i < n:
        if df.iloc[i]['is_bull']:
            j = i + 1
            while j < n and df.iloc[j]['is_bull']:
                j += 1
            count = j - i
            if count >= min_candles:
                seg = df.iloc[i:j]
                move = seg['close'].iloc[-1] - seg['open'].iloc[0]
                if move >= atr_mult * seg['atr'].mean():
                    displacements.append(Displacement(
                        i, j-1, 'UP', move, count,
                        seg['open'].iloc[0], seg['close'].iloc[-1],
                        seg['time'].iloc[0], seg['time'].iloc[-1]))
            i = j
        elif df.iloc[i]['is_bear']:
            j = i + 1
            while j < n and df.iloc[j]['is_bear']:
                j += 1
            count = j - i
            if count >= min_candles:
                seg = df.iloc[i:j]
                move = seg['open'].iloc[0] - seg['close'].iloc[-1]
                if move >= atr_mult * seg['atr'].mean():
                    displacements.append(Displacement(
                        i, j-1, 'DOWN', move, count,
                        seg['open'].iloc[0], seg['close'].iloc[-1],
                        seg['time'].iloc[0], seg['time'].iloc[-1]))
            i = j
        else:
            i += 1
    return displacements


displacements = detect_displacements(df_m5)
up_d   = [d for d in displacements if d.direction == 'UP']
down_d = [d for d in displacements if d.direction == 'DOWN']
print(f'Displacements: {len(displacements)}  (UP={len(up_d)}, DOWN={len(down_d)})')
if displacements:
    moves = [d.total_move for d in displacements]
    print(f'  Avg move : {np.mean(moves):,.2f}  |  Max: {max(moves):,.2f}')

## Step 5 — Order Block Identification

In [ ]:
@dataclass
class OrderBlock:
    disp_idx:    int
    ob_bar_idx:  int
    ob_type:     str
    ob_high:     float
    ob_low:      float
    ob_time:     object
    disp_dir:    str
    displaced_by: float


def find_order_blocks(df: pd.DataFrame, displacements: List[Displacement]) -> List[OrderBlock]:
    order_blocks = []
    for disp_num, disp in enumerate(displacements):
        start_i = disp.start_idx
        look_back = max(0, start_i - 20)
        if disp.direction == 'UP':
            for k in range(start_i - 1, look_back - 1, -1):
                if df.iloc[k]['is_bear']:
                    bar = df.iloc[k]
                    order_blocks.append(OrderBlock(
                        disp_num, k, 'bullish',
                        bar['high'], bar['low'], bar['time'],
                        'UP', disp.total_move))
                    break
        else:
            for k in range(start_i - 1, look_back - 1, -1):
                if df.iloc[k]['is_bull']:
                    bar = df.iloc[k]
                    order_blocks.append(OrderBlock(
                        disp_num, k, 'bearish',
                        bar['high'], bar['low'], bar['time'],
                        'DOWN', disp.total_move))
                    break
    return order_blocks


order_blocks = find_order_blocks(df_m5, displacements)
bull_obs = [ob for ob in order_blocks if ob.ob_type == 'bullish']
bear_obs = [ob for ob in order_blocks if ob.ob_type == 'bearish']
print(f'Order Blocks: {len(order_blocks)}  (Bullish={len(bull_obs)}, Bearish={len(bear_obs)})')

## Step 6 — FVG Detection

In [ ]:
def detect_fvg(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['fvg_type']     = None
    df['fvg_top']      = np.nan
    df['fvg_bottom']   = np.nan
    df['fvg_midpoint'] = np.nan
    for i in range(2, len(df)):
        h2 = df.iloc[i-2]['high']
        l2 = df.iloc[i-2]['low']
        lc = df.iloc[i]['low']
        hc = df.iloc[i]['high']
        if h2 < lc:
            df.at[i, 'fvg_type']     = 'bullish'
            df.at[i, 'fvg_top']      = lc
            df.at[i, 'fvg_bottom']   = h2
            df.at[i, 'fvg_midpoint'] = (lc + h2) / 2
        elif l2 > hc:
            df.at[i, 'fvg_type']     = 'bearish'
            df.at[i, 'fvg_top']      = l2
            df.at[i, 'fvg_bottom']   = hc
            df.at[i, 'fvg_midpoint'] = (l2 + hc) / 2
    return df


df_m5 = detect_fvg(df_m5)
print(f'FVG — Bullish: {(df_m5["fvg_type"]=="bullish").sum()}  '
      f'Bearish: {(df_m5["fvg_type"]=="bearish").sum()}')

## Step 7 — Backtesting Engine

In [ ]:
def simulate_trade(df, entry_idx, direction, entry, sl, tp):
    for i in range(entry_idx, min(entry_idx + MAX_TRADE_BARS, len(df))):
        bar = df.iloc[i]
        if direction == 'BUY':
            if bar['low']  <= sl: return 'SL', -1.0, i - entry_idx + 1
            if bar['high'] >= tp: return 'TP', float(RISK_REWARD), i - entry_idx + 1
        else:
            if bar['high'] >= sl: return 'SL', -1.0, i - entry_idx + 1
            if bar['low']  <= tp: return 'TP', float(RISK_REWARD), i - entry_idx + 1
    return 'OPEN', 0.0, MAX_TRADE_BARS


def is_rejection_candle(bar, ob_type):
    if bar['range'] == 0:
        return False
    if ob_type == 'bullish':
        return bar['lower_wick'] / bar['range'] > REJECTION_WICK_RATIO
    return bar['upper_wick'] / bar['range'] > REJECTION_WICK_RATIO


def run_ob_backtest(df, order_blocks):
    trades = []
    n = len(df)
    for ob in order_blocks:
        displaced = False
        for i in range(ob.ob_bar_idx + 1, min(ob.ob_bar_idx + OB_EXPIRY_BARS, n - 1)):
            bar = df.iloc[i]
            if ob.ob_type == 'bullish':
                if not displaced:
                    if bar['close'] > ob.ob_high: displaced = True
                    continue
                if bar['low'] <= ob.ob_high and bar['close'] >= ob.ob_low:
                    if is_rejection_candle(bar, ob.ob_type):
                        entry = ob.ob_high
                        sl    = ob.ob_low - SL_BUFFER
                        tp    = entry + (entry - sl) * RISK_REWARD
                        result, pnl_r, bars = simulate_trade(df, i+1, 'BUY', entry, sl, tp)
                        fvg_conf = bool((df.iloc[max(0,ob.ob_bar_idx-10):i+1]['fvg_type']=='bullish').any())
                        trades.append({'direction':'BUY','entry_price':entry,'sl':sl,'tp':tp,
                                       'result':result,'pnl_r':pnl_r,'bars_held':bars,
                                       'retest_time':bar['time'],'ob_time':ob.ob_time,
                                       'ob_type':ob.ob_type,'fvg_confluence':fvg_conf})
                        break
                if bar['close'] < ob.ob_low: break
            else:
                if not displaced:
                    if bar['close'] < ob.ob_low: displaced = True
                    continue
                if bar['high'] >= ob.ob_low and bar['close'] <= ob.ob_high:
                    if is_rejection_candle(bar, ob.ob_type):
                        entry = ob.ob_low
                        sl    = ob.ob_high + SL_BUFFER
                        tp    = entry - (sl - entry) * RISK_REWARD
                        result, pnl_r, bars = simulate_trade(df, i+1, 'SELL', entry, sl, tp)
                        fvg_conf = bool((df.iloc[max(0,ob.ob_bar_idx-10):i+1]['fvg_type']=='bearish').any())
                        trades.append({'direction':'SELL','entry_price':entry,'sl':sl,'tp':tp,
                                       'result':result,'pnl_r':pnl_r,'bars_held':bars,
                                       'retest_time':bar['time'],'ob_time':ob.ob_time,
                                       'ob_type':ob.ob_type,'fvg_confluence':fvg_conf})
                        break
                if bar['close'] > ob.ob_high: break
    return pd.DataFrame(trades)


trades_df = run_ob_backtest(df_m5, order_blocks)
if trades_df.empty:
    print('No trades generated.')
else:
    print(f'Total trades : {len(trades_df)}')
    print(f'  BUY / SELL : {(trades_df["direction"]=="BUY").sum()} / {(trades_df["direction"]=="SELL").sum()}')
    print(f'  TP  / SL   : {(trades_df["result"]=="TP").sum()} / {(trades_df["result"]=="SL").sum()}')
    display(trades_df.head(5))

## Step 8 — Performance Analytics

In [ ]:
def calc_metrics(trades_df: pd.DataFrame) -> dict:
    if trades_df.empty:
        return {}
    closed = trades_df[trades_df['result'].isin(['TP','SL'])].copy()
    if closed.empty:
        return {}
    n    = len(closed)
    wins = (closed['result'] == 'TP').sum()
    wr   = wins / n
    closed['cum_r'] = closed['pnl_r'].cumsum()
    dd   = closed['cum_r'] - closed['cum_r'].cummax()
    pos  = closed[closed['pnl_r'] > 0]['pnl_r'].sum()
    neg  = abs(closed[closed['pnl_r'] < 0]['pnl_r'].sum())
    pf   = pos / neg if neg > 0 else float('inf')
    arr  = (closed['result']=='SL').astype(int).values
    max_cl = streak = 0
    for v in arr:
        streak = (streak+1) if v else 0
        max_cl = max(max_cl, streak)
    return {
        'total_trades': n, 'wins': int(wins), 'losses': n-int(wins),
        'win_rate': wr, 'total_r': round(closed['pnl_r'].sum(), 3),
        'avg_r': round(closed['pnl_r'].mean(), 3),
        'profit_factor': round(pf, 3), 'max_dd_r': round(dd.min(), 3),
        'max_consec_loss': max_cl,
        'expectancy': round(wr*RISK_REWARD - (1-wr), 3),
        'avg_bars': round(closed['bars_held'].mean(), 1),
        'cum_r': closed['cum_r'].reset_index(drop=True),
        'drawdown': dd.reset_index(drop=True),
        'closed': closed,
    }


metrics = calc_metrics(trades_df)
if metrics and metrics.get('total_trades', 0) > 0:
    sep = '=' * 58
    print(sep)
    print(f' {SYMBOL} — Order Block Reaction | {LOOKBACK_DAYS}d backtest')
    print(sep)
    print(f'  Trades        : {metrics["total_trades"]}')
    print(f'  Win Rate      : {metrics["win_rate"]*100:.1f}%  ({metrics["wins"]}W / {metrics["losses"]}L)')
    print(f'  Total R       : {metrics["total_r"]:+.2f} R')
    print(f'  Avg R/trade   : {metrics["avg_r"]:+.3f} R')
    print(f'  Profit Factor : {metrics["profit_factor"]:.2f}')
    print(f'  Expectancy    : {metrics["expectancy"]:+.3f} R')
    print(f'  Max Drawdown  : {metrics["max_dd_r"]:.2f} R')
    print(f'  Max Consec SL : {metrics["max_consec_loss"]}')
    print(f'  Avg Duration  : {metrics["avg_bars"]:.0f} M5 bars')
    print(sep)

## Step 8.5 — Fee-Adjusted Performance (Crypto)

محاسبه سود/ضرر واقعی پس از کسر فی بر اساس **درصد Notional Value**.

### فرمول محاسبه
```
entry_fee = entry_price × quantity × entry_fee_pct
exit_fee  = exit_price  × quantity × exit_fee_pct
total_fee = entry_fee + exit_fee

Gross$ = pnl_r × dollar_risk
Net$   = Gross$ − total_fee
```

> نکته مهم: exit_price برای TP و SL متفاوت است، بنابراین fee هر ترید متفاوت است.

In [ ]:
def crypto_fee_analysis(trades_df: pd.DataFrame, scenario_name: str, fee: dict) -> pd.DataFrame:
    """Per-trade net P&L after crypto maker/taker fees."""
    closed = trades_df[trades_df['result'].isin(['TP','SL'])].copy()
    if closed.empty:
        return pd.DataFrame()

    # Actual exit price for each trade
    closed['exit_price'] = np.where(closed['result'] == 'TP', closed['tp'], closed['sl'])

    # Notional values in USDT
    closed['notional_entry'] = closed['entry_price'] * QUANTITY
    closed['notional_exit']  = closed['exit_price']  * QUANTITY

    # Fee per trade (varies because exit_price differs per trade)
    closed['entry_fee_usd'] = closed['notional_entry'] * fee['entry_pct']
    closed['exit_fee_usd']  = closed['notional_exit']  * fee['exit_pct']
    closed['total_cost_usd']= closed['entry_fee_usd']  + closed['exit_fee_usd']

    # Entry fee % and exit fee %
    closed['entry_fee_pct'] = fee['entry_pct'] * 100
    closed['exit_fee_pct']  = fee['exit_pct']  * 100

    # Gross P&L from backtest R values
    closed['dollar_risk']   = abs(closed['entry_price'] - closed['sl']) * QUANTITY
    closed['pnl_usd_gross'] = closed['pnl_r'] * closed['dollar_risk']

    # Net P&L
    closed['pnl_usd_net']   = closed['pnl_usd_gross'] - closed['total_cost_usd']
    closed['cum_usd_gross'] = closed['pnl_usd_gross'].cumsum()
    closed['cum_usd_net']   = closed['pnl_usd_net'].cumsum()

    # R-based (cost_r varies per trade because fee is % of notional)
    closed['cost_r']    = closed['total_cost_usd'] / closed['dollar_risk']
    closed['pnl_r_net'] = closed['pnl_r'] - closed['cost_r']
    closed['cum_r_net'] = closed['pnl_r_net'].cumsum()

    closed['scenario'] = scenario_name
    closed['quantity'] = QUANTITY
    return closed


fee_results = {
    name: crypto_fee_analysis(trades_df, name, params)
    for name, params in FEE_SCENARIOS.items()
}

# ── Cost breakdown info ───────────────────────────────────────────────────────
last_price = df_m5['close'].iloc[-1]
print(f'{SYMBOL} last price : ${last_price:,.2f}')
print(f'Notional for {QUANTITY} coin : ${QUANTITY * last_price:,.2f} USDT')
print()
print('FEE PER TRADE (estimated at current price):')
print(f'  {"Scenario":<16} {"Entry fee":>12} {"Exit fee":>12} {"RT Total":>12}')
print(f'  {"-"*50}')
for name, fee in FEE_SCENARIOS.items():
    ef  = QUANTITY * last_price * fee['entry_pct']
    xf  = QUANTITY * last_price * fee['exit_pct']
    print(f'  {name:<16} ${ef:>10.4f}  ${xf:>10.4f}  ${ef+xf:>10.4f}')

# ── Summary table ─────────────────────────────────────────────────────────────
sep = '=' * 105
print(f'\n{sep}')
print(f'  CRYPTO FEE-ADJUSTED PERFORMANCE  |  {SYMBOL}  |  Qty: {QUANTITY}  |  Capital: ${INITIAL_CAPITAL:,}')
print(sep)
print(f'  {"Scenario":<16} {"AvgFee$/tr":>11} {"TotFee$":>9} {"Gross$":>9} '
      f'{"Net$":>9} {"Return%":>9} {"WR%":>7} {"PF(net)":>8}')
print(f'  {"-"*93}')

for scenario_name, df_s in fee_results.items():
    if df_s.empty:
        continue
    n        = len(df_s)
    avg_fee  = df_s['total_cost_usd'].mean()
    tot_fee  = df_s['total_cost_usd'].sum()
    gross    = df_s['pnl_usd_gross'].sum()
    net_d    = df_s['pnl_usd_net'].sum()
    ret_pct  = net_d / INITIAL_CAPITAL * 100
    wr       = (df_s['result'] == 'TP').mean() * 100
    pos      = df_s[df_s['pnl_usd_net'] > 0]['pnl_usd_net'].sum()
    neg      = abs(df_s[df_s['pnl_usd_net'] < 0]['pnl_usd_net'].sum())
    pf       = pos / neg if neg > 0 else float('inf')
    sign     = '+' if ret_pct >= 0 else ''
    print(f'  {scenario_name:<16} {avg_fee:>11.4f} {tot_fee:>9.4f} {gross:>9.4f} '
          f'{net_d:>9.4f} {sign}{ret_pct:>7.4f}% {wr:>7.1f}% {pf:>8.2f}')

print(sep)
print(f'  Return% = Net$ / ${INITIAL_CAPITAL:,}  |  Fee varies per trade (% of exit/entry notional)')

best = max(fee_results, key=lambda k: fee_results[k]['pnl_usd_net'].sum()
           if not fee_results[k].empty else -999)
bd = fee_results[best]
print(f'\n  Best scenario : {best}  →  Net: ${bd["pnl_usd_net"].sum():.4f}  '
      f'| Avg fee/trade: ${bd["total_cost_usd"].mean():.4f}')

### Per-Trade Fee Breakdown

In [ ]:
# Per-trade table: all 3 scenarios side-by-side
mm_df  = fee_results['Maker-Maker']
mt_df  = fee_results['Maker-Taker']
tt_df  = fee_results['Taker-Taker']

sep = '=' * 125
print(sep)
print(f'  PER-TRADE BREAKDOWN  |  {SYMBOL}  |  Qty: {QUANTITY}')
print(f'  Fee varies per trade — exit price differs for TP vs SL')
print(sep)
print(f'  {"#":>3} {"Dir":>5} {"Entry":>12} {"Res":>5} {"Risk$":>8} {"Gross$":>9} '
      f'| {"MM Fee$":>9} {"MM Net$":>9} '
      f'| {"MT Fee$":>9} {"MT Net$":>9} '
      f'| {"TT Fee$":>9} {"TT Net$":>9} '
      f'| {"CumNet(MT)$":>12}')
print(f'  {"-"*122}')

for idx in range(len(mt_df)):
    r_mm  = mm_df.iloc[idx]
    r_mt  = mt_df.iloc[idx]
    r_tt  = tt_df.iloc[idx]
    gross = r_mt['pnl_usd_gross']
    res   = r_mt['result']
    print(f'  {idx+1:>3} {r_mt["direction"]:>5} {r_mt["entry_price"]:>12,.2f} {res:>5} '
          f'{r_mt["dollar_risk"]:>8.4f} {gross:>+9.4f} '
          f'| {r_mm["total_cost_usd"]:>9.4f} {r_mm["pnl_usd_net"]:>+9.4f} '
          f'| {r_mt["total_cost_usd"]:>9.4f} {r_mt["pnl_usd_net"]:>+9.4f} '
          f'| {r_tt["total_cost_usd"]:>9.4f} {r_tt["pnl_usd_net"]:>+9.4f} '
          f'| {r_mt["cum_usd_net"]:>+12.4f}')

print(f'  {"-"*122}')
print(f'  {"TOTAL":>20} {"":>8} {mt_df["pnl_usd_gross"].sum():>+9.4f} '
      f'| {mm_df["total_cost_usd"].sum():>9.4f} {mm_df["pnl_usd_net"].sum():>+9.4f} '
      f'| {mt_df["total_cost_usd"].sum():>9.4f} {mt_df["pnl_usd_net"].sum():>+9.4f} '
      f'| {tt_df["total_cost_usd"].sum():>9.4f} {tt_df["pnl_usd_net"].sum():>+9.4f} '
      f'| {mt_df["cum_usd_net"].iloc[-1]:>+12.4f}')
print(sep)
print(f'  MM=Maker-Maker (both limit)  MT=Maker-Taker (limit in, market out)  TT=Taker-Taker (both market)')
print(f'  Fee$ = entry_fee + exit_fee  (exit_fee differs for TP vs SL because exit price differs)')

## Step 8.6 — Save Trades to Files

In [ ]:
OUT_BASE = RESULT_DIR / '08_order_block_reaction_crypto' / SYMBOL
OUT_BASE.mkdir(parents=True, exist_ok=True)

trades_df.to_csv(OUT_BASE / 'trades_raw.csv', index=False)
print(f'  trades_raw.csv  ({len(trades_df)} rows)')

SAVE_COLS = [
    'scenario', 'quantity', 'direction', 'ob_type', 'fvg_confluence',
    'retest_time', 'ob_time', 'entry_price', 'exit_price', 'sl', 'tp',
    'result', 'bars_held', 'pnl_r',
    'dollar_risk',
    'notional_entry', 'notional_exit',
    'entry_fee_pct', 'entry_fee_usd',
    'exit_fee_pct',  'exit_fee_usd',
    'total_cost_usd',
    'pnl_usd_gross', 'pnl_usd_net', 'cum_usd_gross', 'cum_usd_net',
    'cost_r', 'pnl_r_net', 'cum_r_net',
]

summary_rows = []
for scenario_name, df_s in fee_results.items():
    if df_s.empty:
        continue
    fee   = FEE_SCENARIOS[scenario_name]
    s_dir = OUT_BASE / scenario_name.replace('-', '_').replace(' ', '_')
    s_dir.mkdir(parents=True, exist_ok=True)

    out_cols = [c for c in SAVE_COLS if c in df_s.columns]
    df_s[out_cols].to_csv(s_dir / 'trades.csv', index=False)

    n   = len(df_s)
    pos = df_s[df_s['pnl_usd_net'] > 0]['pnl_usd_net'].sum()
    neg = abs(df_s[df_s['pnl_usd_net'] < 0]['pnl_usd_net'].sum())
    pf  = round(pos / neg, 4) if neg > 0 else None
    dd  = (df_s['cum_usd_net'] - df_s['cum_usd_net'].cummax()).min()

    print(f'  {scenario_name}/trades.csv  '
          f'avg_fee: ${df_s["total_cost_usd"].mean():.4f}  '
          f'net: ${df_s["pnl_usd_net"].sum():.4f}')

    summary_rows.append({
        'scenario'             : scenario_name,
        'quantity'             : QUANTITY,
        'entry_fee_pct'        : fee['entry_pct'] * 100,
        'exit_fee_pct'         : fee['exit_pct']  * 100,
        'rt_fee_pct'           : (fee['entry_pct'] + fee['exit_pct']) * 100,
        'avg_fee_per_trade_usd': round(df_s['total_cost_usd'].mean(), 6),
        'total_fee_paid_usd'   : round(df_s['total_cost_usd'].sum(), 6),
        'total_trades'         : n,
        'wins'                 : int((df_s['result'] == 'TP').sum()),
        'losses'               : int((df_s['result'] == 'SL').sum()),
        'win_rate'             : round((df_s['result'] == 'TP').mean(), 4),
        'gross_total_usd'      : round(df_s['pnl_usd_gross'].sum(), 6),
        'net_total_usd'        : round(df_s['pnl_usd_net'].sum(), 6),
        'net_total_r'          : round(df_s['pnl_r_net'].sum(), 4),
        'return_pct'           : round(df_s['pnl_usd_net'].sum() / INITIAL_CAPITAL * 100, 6),
        'max_drawdown_usd'     : round(dd, 6),
        'profit_factor_net'    : pf,
    })

metrics_summary = pd.DataFrame(summary_rows)
metrics_summary.to_csv(OUT_BASE / 'metrics_by_scenario.csv', index=False)
print(f'\n  metrics_by_scenario.csv')
print(f'  All files → {OUT_BASE}\n')
display(metrics_summary[[
    'scenario', 'entry_fee_pct', 'exit_fee_pct', 'rt_fee_pct',
    'avg_fee_per_trade_usd', 'total_fee_paid_usd',
    'gross_total_usd', 'net_total_usd', 'return_pct'
]])

## Step 8.7 — Capital & Leverage Simulation

شبیه‌سازی با **سرمایه واقعی** و **اهرم** در صرافی کریپتو.

| سناریو ریسک | تعریف |
|------------|-------|
| Conservative (1%) | quantity = 1% سرمایه / dollar_risk_per_coin |
| Moderate (2%) | quantity = 2% سرمایه / dollar_risk_per_coin |
| Max Margin (80%) | 80% سرمایه به عنوان مارجین |

In [ ]:
LEVERAGE = 10   # change to match your exchange setting

last_price    = df_m5['close'].iloc[-1]
notional_1qty = last_price * 1             # notional for 1 coin
margin_1qty   = notional_1qty / LEVERAGE   # margin needed for 1 coin

closed_t = trades_df[trades_df['result'].isin(['TP','SL'])].copy()
closed_t['sl_dist'] = abs(closed_t['entry_price'] - closed_t['sl'])
avg_sl = closed_t['sl_dist'].mean()

qty_1pct = (INITIAL_CAPITAL * 0.01) / (avg_sl)     # risk $= 1% capital
qty_2pct = (INITIAL_CAPITAL * 0.02) / (avg_sl)
max_margin_qty = (INITIAL_CAPITAL * 0.80) / margin_1qty

def snapq(x, min_q=0.0001): return max(min_q, round(x, 4))

scenarios = {
    'Conservative (1% risk)': min(snapq(qty_1pct),    snapq(max_margin_qty)),
    'Moderate     (2% risk)': min(snapq(qty_2pct),    snapq(max_margin_qty)),
    'Max Margin   (80% cap)': snapq(max_margin_qty),
}

sep = '=' * 68
print(sep)
print(f'  MARGIN CALCULATOR  |  Capital: ${INITIAL_CAPITAL:,}  |  Leverage: {LEVERAGE}x')
print(sep)
print(f'  {SYMBOL} last price     : ${last_price:,.2f}')
print(f'  Notional for 1 coin  : ${notional_1qty:,.2f}')
print(f'  Margin for 1 coin    : ${margin_1qty:,.2f}')
print(f'  Avg SL distance      : ${avg_sl:,.2f}')
print(f'  Qty for 1% risk      : {snapq(qty_1pct):.4f} coins')
print(f'  Qty for 2% risk      : {snapq(qty_2pct):.4f} coins')
print(f'  Max qty (80% margin) : {snapq(max_margin_qty):.4f} coins')
print(sep)

# Best fee scenario for scaling
best_scenario = 'Maker-Taker'
base_df = fee_results[best_scenario]

print(f'\nSCALED P&L  ({best_scenario} fees)  |  Capital ${INITIAL_CAPITAL:,} @ {LEVERAGE}x')
for scen_name, qty in scenarios.items():
    scale       = qty / QUANTITY
    margin_used = margin_1qty * qty
    margin_pct  = margin_used / INITIAL_CAPITAL * 100
    print(f'\n  ── {scen_name}  |  Qty: {qty:.4f}  |  Margin: ${margin_used:,.2f} ({margin_pct:.0f}%)')
    print(f'  {"Scenario":<16} {"AvgFee$/tr":>12} {"Net$/period":>13} '
          f'{"NetR":>8} {"MaxDD$":>9} {"Return%":>9} {"Ann%*":>8}')
    print(f'  {"-"*80}')
    for scen, df_s in fee_results.items():
        if df_s.empty:
            continue
        net_usd = df_s['pnl_usd_net'].sum() * scale
        net_r   = df_s['pnl_r_net'].sum()
        max_dd  = (df_s['cum_usd_net'] - df_s['cum_usd_net'].cummax()).min() * scale
        avg_fee = df_s['total_cost_usd'].mean() * scale
        ret_pct = net_usd / INITIAL_CAPITAL * 100
        ann_pct = ret_pct * (252 / 5)
        print(f'  {scen:<16} {avg_fee:>12.4f} {net_usd:>+13.4f} '
              f'{net_r:>8.2f} {max_dd:>9.4f} {ret_pct:>8.4f}% {ann_pct:>7.0f}%')

print(f'\n  * Ann% = 5-day result × 252/5  (theoretical)')
print(f'  * Return% = Net$ / ${INITIAL_CAPITAL:,}')

## Step 8.8 — Money Management Strategy Comparison

In [ ]:
import math

BASE_QTY        = QUANTITY
WIN_RATE        = (fee_results['Maker-Taker']['result'] == 'TP').mean()
KELLY_PCT       = max(0.0, WIN_RATE - (1 - WIN_RATE) / RISK_REWARD)
HALF_KELLY      = KELLY_PCT / 2

base_df = fee_results['Maker-Taker'].copy().reset_index(drop=True)
base_df['sl_dist'] = base_df['dollar_risk'] / BASE_QTY

def snapq(x): return max(0.0001, round(x, 4))

def strat_fixed_base(eq, streak_w, streak_l, i):      return BASE_QTY
def strat_fixed_1pct(eq, streak_w, streak_l, i):
    dr = base_df.iloc[i]['sl_dist']
    return snapq((INITIAL_CAPITAL * 0.01) / dr) if dr > 0 else BASE_QTY
def strat_compound_1pct(eq, streak_w, streak_l, i):
    dr = base_df.iloc[i]['sl_dist']
    return snapq((eq * 0.01) / dr) if dr > 0 else BASE_QTY
def strat_half_kelly(eq, streak_w, streak_l, i):
    dr = base_df.iloc[i]['sl_dist']
    return snapq((eq * HALF_KELLY) / dr) if dr > 0 else BASE_QTY
def strat_anti_martingale(eq, streak_w, streak_l, i):
    if streak_w == 0: return BASE_QTY
    return snapq(BASE_QTY * min(1.5 ** min(streak_w, 3), 1.5**3))
def strat_staged(eq, streak_w, streak_l, i):
    steps = max(0, int((eq - INITIAL_CAPITAL) // 500))
    return snapq(BASE_QTY + steps * BASE_QTY)

STRATEGIES = {
    'Fixed Base'        : strat_fixed_base,
    'Fixed 1% Risk'     : strat_fixed_1pct,
    'Compound 1%'       : strat_compound_1pct,
    'Half-Kelly'        : strat_half_kelly,
    'Anti-Martingale'   : strat_anti_martingale,
    'Staged +qty/$500'  : strat_staged,
}

def simulate_mm(trades, strategy_fn):
    equity = INITIAL_CAPITAL
    peak   = INITIAL_CAPITAL
    max_dd = 0.0
    curve  = [equity]
    sw = sl = 0
    for i, row in trades.iterrows():
        qty    = strategy_fn(equity, sw, sl, i)
        scale  = qty / BASE_QTY
        pnl    = row['pnl_usd_net'] * scale
        equity = max(0.0, equity + pnl)
        peak   = max(peak, equity)
        max_dd = max(max_dd, peak - equity)
        if row['result'] == 'TP': sw += 1; sl = 0
        else:                     sl += 1; sw = 0
        curve.append(equity)
    return {
        'curve': curve, 'final': equity,
        'net': equity - INITIAL_CAPITAL,
        'ret': (equity - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100,
        'max_dd': max_dd, 'max_dd_pct': max_dd / INITIAL_CAPITAL * 100,
    }

results = {name: simulate_mm(base_df, fn) for name, fn in STRATEGIES.items()}

sep = '=' * 85
print(sep)
print(f'  MM COMPARISON  |  Capital: ${INITIAL_CAPITAL:,}  |  Maker-Taker fees  |  Trades: {len(base_df)}')
print(f'  WR={WIN_RATE*100:.1f}%  RR={RISK_REWARD}  Kelly={KELLY_PCT*100:.2f}%  Half-Kelly={HALF_KELLY*100:.2f}%')
print(sep)
print(f'  {"Strategy":<22} {"Final$":>12} {"Net P&L":>12} {"Return%":>9} {"MaxDD$":>10} {"MaxDD%":>8}')
print(f'  {"-"*75}')
for name, r in results.items():
    s = '+' if r['net'] >= 0 else ''
    print(f'  {name:<22} {r["final"]:>12,.4f} {s}{r["net"]:>11,.4f} '
          f'{s}{r["ret"]:>7.4f}% {r["max_dd"]:>10,.4f} {r["max_dd_pct"]:>7.2f}%')
print(sep)

PALETTE = ['#00E5FF','#00E676','#FFAB40','#7C4DFF','#FF6D00','#F50057']
fig = go.Figure()
for (name, r), color in zip(results.items(), PALETTE):
    fig.add_trace(go.Scatter(
        x=list(range(len(r['curve']))), y=r['curve'],
        mode='lines', name=f'{name}  (${r["final"]:,.2f})',
        line=dict(color=color, width=2)))
fig.add_hline(y=INITIAL_CAPITAL, line_dash='dash', line_color='gray',
              annotation_text=f'Start ${INITIAL_CAPITAL:,}')
fig.update_layout(
    title=f'MM Strategy Comparison — {SYMBOL} | Maker-Taker fees | WR {WIN_RATE*100:.1f}%',
    xaxis_title='Trade #', yaxis_title='Equity (USDT)',
    template='plotly_dark', height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()

## Step 9 — Visualizations
### 9.1 — Equity Curve

In [ ]:
def plot_equity(metrics: dict) -> None:
    if not metrics or 'cum_r' not in metrics:
        return
    cum_r  = metrics['cum_r']
    dd     = metrics['drawdown']
    closed = metrics['closed'].reset_index(drop=True)
    fig = make_subplots(rows=3, cols=1, row_heights=[0.5, 0.25, 0.25],
                        subplot_titles=['Equity (R)', 'Drawdown', 'Per-Trade PnL'],
                        vertical_spacing=0.08)
    fig.add_trace(go.Scatter(
        x=cum_r.index, y=cum_r.values, mode='lines',
        line=dict(color='#00E5FF', width=2.5),
        fill='tozeroy', fillcolor='rgba(0,229,255,0.08)', name='Equity'), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=cum_r.index, y=cum_r.cummax().values, mode='lines',
        line=dict(color='gold', width=1, dash='dot'), name='Peak'), row=1, col=1)
    fig.add_hline(y=0, line_color='gray', line_dash='dash', row=1, col=1)
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd.values, mode='lines',
        fill='tozeroy', fillcolor='rgba(255,23,68,0.15)',
        line=dict(color='#FF1744', width=1.5), name='DD'), row=2, col=1)
    colors = ['#00E676' if r == 'TP' else '#FF1744' for r in closed['result']]
    fig.add_trace(go.Bar(x=closed.index, y=closed['pnl_r'], marker_color=colors, name='PnL'), row=3, col=1)
    fig.add_hline(y=0, line_color='gray', line_dash='dash', row=3, col=1)
    fig.update_layout(
        title=f'Order Block — {SYMBOL} Equity | WR={metrics["win_rate"]*100:.0f}% Total={metrics["total_r"]:+.1f}R',
        height=700, template='plotly_dark')
    fig.show()

plot_equity(metrics)

### 9.2 — Fee Comparison Chart

In [ ]:
fig = go.Figure()
colors = ['#00E5FF', '#FFAB40', '#FF1744']
for (scenario_name, df_s), color in zip(fee_results.items(), colors):
    if df_s.empty:
        continue
    fig.add_trace(go.Scatter(
        x=list(range(len(df_s))),
        y=df_s['cum_usd_net'].values,
        mode='lines',
        name=f'{scenario_name}  (${df_s["pnl_usd_net"].sum():.4f})',
        line=dict(color=color, width=2)))

fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(
    title=f'Cumulative Net P&L by Fee Scenario — {SYMBOL} | Qty: {QUANTITY}',
    xaxis_title='Trade #', yaxis_title='Cumulative Net P&L (USDT)',
    template='plotly_dark', height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()

## Final Analysis

In [ ]:
if metrics and metrics.get('total_trades', 0) > 0:
    be_wr = 1 / (1 + RISK_REWARD)
    last_price = df_m5['close'].iloc[-1]
    print('=' * 62)
    print(f'  ORDER BLOCK REACTION CRYPTO — FINAL ANALYSIS  ({SYMBOL})')
    print('=' * 62)
    print(f'\n[EDGE (gross, no fees)]')
    print(f'  Break-even WR : {be_wr*100:.1f}%')
    print(f'  Actual WR     : {metrics["win_rate"]*100:.1f}%')
    print(f'  Expectancy    : {metrics["expectancy"]:+.3f} R')
    edge = 'POSITIVE' if metrics['win_rate'] > be_wr else 'NEGATIVE'
    print(f'  Edge          : {edge}')

    print(f'\n[CRYPTO FEE IMPACT] (qty={QUANTITY}, price~${last_price:,.0f})')
    notional = QUANTITY * last_price
    print(f'  Notional/trade   : ~${notional:,.2f} USDT')
    for name, fee in FEE_SCENARIOS.items():
        rt = (fee['entry_pct'] + fee['exit_pct']) * 100
        rt_usd = notional * (fee['entry_pct'] + fee['exit_pct'])
        print(f'  {name:<16}: {rt:.3f}% RT  ≈  ${rt_usd:.4f}/trade')

    print(f'\n[CRYPTO vs FOREX DIFFERENCES]')
    print('  ✓ Fee is % of notional — scales with price, not fixed')
    print('  ✓ Exit price (TP vs SL) affects exit fee amount')
    print('  ✓ Maker-Taker model incentivizes limit orders')
    print('  ✗ No spread concept — exchange uses order book')
    print('  ✗ Funding rates apply for perpetual futures (not modeled)')
    print('  ✗ Slippage can be higher in thin crypto markets')

    print(f'\n[OPTIMIZATIONS FOR CRYPTO]')
    print('  → Use limit orders (maker fee) to reduce cost')
    print('  → Filter by high-volume sessions (US/Asia open)')
    print('  → Apply volume filter — only trade high liquidity candles')
    print('  → Consider funding rate cost for overnight positions')
    print('  → Use larger quantity to make % fees meaningful vs P&L')
    print('=' * 62)